In [1]:
import json
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

### Setup (Loading data, minor preprocesing)

In [2]:
import json
import pandas as pd

RESOLUTIONS = ['50', '71', '87', '100']
PARAMS = ['alpha_weight', 'hist_percent', 'filter_size', 'num_samples']

# Scenes to exclude (outliers)
EXCLUDE_SCENES = [
    'junkyard-mound1', 'junkyard-mound2',
    'oldmine-speed-18', 'oldmine-speed-35',
    'oldmine-speed-75', 'oldmine-speed-9', 'oldmine-warm', 
    'wildwest-barzoom'
]

with open('dataset.json', 'r') as f:
    raw = json.load(f)

records = []
for scene, scene_data in raw.items():
    if not isinstance(scene_data, dict):
        continue
    for res in RESOLUTIONS:
        if res not in scene_data:
            continue
        for ref, ref_data in scene_data[res].items():
            if ref != f'ref-{scene}': # we only want data when lower resolutions are compared to full-quality 16xSSAA
                continue
            for param in PARAMS:
                if param not in ref_data:
                    continue
                for entry in ref_data[param]:
                    records.append({
                        'scene':            scene,
                        'resolution':       int(res),
                        'parameter':        param,
                        'value':            entry['value'],
                        'score':            entry['score'],
                        'per_frame_errors': entry.get('per_frame_errors', []), # this is optional, can ignore
                    })

df = pd.DataFrame(records)
df = df[~df['scene'].isin(EXCLUDE_SCENES)]

# Exclude hist_percent values below 100 (must do because Unreal treats all sub 100 values as 100)
df = df[~((df['parameter'] == 'hist_percent') & (df['value'] < 100))]

In [3]:
# find which values are shared across all scenes for each (parameter, resolution) 
common_vals_count = df.groupby(['parameter', 'resolution', 'value'])['scene'].nunique()
n_scenes_per_param_res = df.groupby(['parameter','resolution'])['scene'].nunique()

def center_score(group):
	# group is a subset of df for one specific (scene, parameter, resolution) combination 
	# e.g. all rows for "abandoned" + "alpha_weight" + resolution 50
    param, res = group['parameter'].iloc[0], group['resolution'].iloc[0]
    
    # get the values tested for this (parameter, resolution) across all scenes
    common = df[(df['parameter'] == param) & (df['resolution'] == res)]
    common_vals = common.groupby('value')['scene'].nunique()
    n_scenes = common['scene'].nunique()
    shared_vals = common_vals[common_vals == n_scenes].index
    
    # compute mean score over shared values only, then subtract from all scores
    mask = group['value'].isin(shared_vals)
    mean = group.loc[mask, 'score'].mean()
    return group['score'] - mean

df['score_centered'] = df.groupby(
    ['scene', 'parameter', 'resolution'], group_keys=False
).apply(center_score)

/var/folders/wh/dwptv6ss1nzbsg6xw_2p4c9h0000gn/T/ipykernel_17997/1648496047.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(center_score)


### Basic Info about Data

In [4]:
df.groupby(['parameter', 'resolution'])['score_centered'].agg(['count', 'mean', 'std']).round(3)

count  mean    std
parameter    resolution                    
alpha_weight 50            240   0.0  3.033
             71            240  -0.0  1.468
             87            240   0.0  1.209
             100           240   0.0  1.341
filter_size  50            140   0.0  0.094
             71            140  -0.0  0.109
             87            140   0.0  0.131
             100           140   0.0  0.130
hist_percent 50             80   0.0  0.699
             71             80   0.0  0.671
             87             80   0.0  0.717
             100            80   0.0  0.712
num_samples  50            100   0.0  0.104
             71            100   0.0  0.123
             87            100  -0.0  0.134
             100           100   0.0  0.160

In [5]:
print(f"Scenes: {df['scene'].nunique()}")
print(f"Scene names: {df['scene'].unique().tolist()}")
print(f"\nParameter value counts:")
print(df.groupby(['parameter', 'value'])['scene'].count().rename('n_scenes'))

Scenes: 20
Scene names: ['abandoned', 'abandoned-demo', 'abandoned-flipped', 'cubetest', 'fantasticvillage-open', 'lightfoliage', 'lightfoliage-close', 'oldmine', 'quarry-all', 'quarry-rocksonly', 'resto-close', 'resto-fwd', 'resto-pan', 'scifi', 'subway-lookdown', 'subway-turn', 'wildwest-bar', 'wildwest-behindcounter', 'wildwest-store', 'wildwest-town']

Parameter value counts:
parameter     value 
alpha_weight  0.01      80
              0.02      80
              0.04      80
              0.06      80
              0.10      80
              0.20      80
              0.50      80
              0.60      80
              0.70      80
              0.80      80
              0.90      80
              1.00      80
filter_size   0.10      80
              0.25      80
              0.50      80
              0.70      80
              0.90      80
              0.95      80
              1.00      80
hist_percent  100.00    80
              125.00    80
              150.00    80
  

In [6]:
# Find missing params (ideally none)
for param in PARAMS:
    sub = df[df['parameter'] == param]
    pivot = sub.groupby(['scene', 'resolution', 'value'])['score'].count().unstack('value')
    missing = pivot[pivot.isna().any(axis=1)]
    if not missing.empty:
        print(f"\n=== {param} ===")
        print(missing.to_string())

### Plot of Parameter values x Resolution (both mean centered and not)

In [7]:
# Setup plot color & orders (for consistency)
res_order = ['100%', '87%', '71%', '50%']
res_colors = [ '#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']

color_scale = alt.Scale(domain=res_order, range=res_colors)

In [8]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 2
y_max = 100
y_scale = alt.Scale(domain=[y_min, y_max])

res_order = ['100%', '87%', '71%', '50%']
res_colors = ['#2a7db5', '#2ca05a', '#e07b3d', '#d4504a']
color_scale = alt.Scale(domain=res_order, range=res_colors)

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean CGVQM Score', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared').properties(
    title='TAA Parameter Response Curves (Raw CGVQM)'
)

alt.VConcatChart(...)

In [9]:
agg = df.groupby(['parameter', 'resolution', 'value'])['score_centered'].agg(['mean', 'std', 'count']).reset_index()
agg['ci95'] = 1.96 * (agg['std'] / np.sqrt(agg['count']))
agg['resolution'] = agg['resolution'].astype(str) + '%'

y_min = (agg['mean'] - agg['ci95']).min() - 0.5
y_max = (agg['mean'] + agg['ci95']).max() + 0.5
y_scale = alt.Scale(domain=[y_min, y_max])

charts = []
for param in PARAMS:
    sub = agg[agg['parameter'] == param]
    band = alt.Chart(sub).mark_errorband(opacity=0.2).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Mean Centered CGVQM', scale=y_scale),
        color=alt.Color('resolution:N', title='Base Resolution', scale=color_scale, sort=res_order),
        tooltip=['resolution', 'value', alt.Tooltip('mean:Q', format='.3f'), alt.Tooltip('ci95:Q', format='.3f')]
    )
    charts.append((band + line).properties(title=f'{param}', width=280, height=200))

alt.vconcat(*[alt.hconcat(*charts[:2]), alt.hconcat(*charts[2:])]).resolve_scale(color='shared').properties(
    title='TAA Parameter Response Curves (Mean-Centered CGVQM)'
)

alt.VConcatChart(...)

### Comparing how Alpha Weight Individual Scene shapes compare to the mean curve

In [ ]:
alpha_df = df[df['parameter'] == 'alpha_weight'].copy()
alpha_df['resolution'] = alpha_df['resolution'].astype(str) + '%'

# Per-scene mean score per value
scene_curves = alpha_df.groupby(['scene', 'resolution', 'value'])['score_centered'].mean().reset_index()

# Mean curve across scenes
mean_curve = scene_curves.groupby(['resolution', 'value'])['score_centered'].agg(['mean','std','count']).reset_index()
mean_curve['ci95'] = 1.96 * mean_curve['std'] / np.sqrt(mean_curve['count'])

# Per-scene peak value
peaks = scene_curves.loc[scene_curves.groupby(['scene','resolution'])['score_centered'].idxmax()]
peaks = peaks.rename(columns={'value': 'peak_value'})

y_min = scene_curves['score_centered'].min() - 0.5
y_max = scene_curves['score_centered'].max() + 0.5
y_scale = alt.Scale(domain=[y_min, y_max])


charts = []
for res in res_order:
    sc = scene_curves[scene_curves['resolution'] == res]
    mc = mean_curve[mean_curve['resolution'] == res]
    pk = peaks[peaks['resolution'] == res]
    col = res_colors[res_order.index(res)]

    # Individual scene lines (thin, transparent)
    individual = alt.Chart(sc).mark_line(opacity=0.3, strokeWidth=1).encode(
        x=alt.X('value:Q', title='alpha_weight'),
        y=alt.Y('score_centered:Q', title='Centered CGVQM', scale=y_scale),
        detail='scene:N',
        color=alt.value(col)
    )

    # Mean line
    mean_line = alt.Chart(mc).mark_line(strokeWidth=2.5, point=True).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=y_scale),
        color=alt.value(col)
    )

    # CI band
    band = alt.Chart(mc).mark_errorband(opacity=0.2).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=y_scale),
        yError='ci95:Q',
        color=alt.value(col)
    )

    charts.append((individual + band + mean_line).properties(
        title=f'{res}', width=220, height=180
    ))

alt.hconcat(*charts).properties(title='Alpha Weight: Per-Scene Curves + Mean (by Resolution)')

## Statistical Significance (Mostly Ignore)

### Friedman Test

In [ ]:
from scipy.stats import friedmanchisquare
import itertools

EXCLUDE_HIST = True  # already filtered above

friedman_results = []

for param in PARAMS:
    for res in [50, 71, 87, 100]:
        sub = df[(df['parameter'] == param) & (df['resolution'] == res)]
        
        # Pivot: rows = scenes, columns = parameter values
        pivot = sub.pivot_table(index='scene', columns='value', values='score').dropna()
        
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            print(f"Skipping {param} @ {res}%: only {pivot.shape[0]} complete scenes, {pivot.shape[1]} levels")
            continue
        
        stat, p = friedmanchisquare(*[pivot[col].values for col in pivot.columns])
        
        friedman_results.append({
            'parameter':  param,
            'resolution': f'{res}%',
            'n_scenes':   pivot.shape[0],
            'n_levels':   pivot.shape[1],
            'statistic':  round(stat, 3),
            'p_value':    p,
            'p_display':  f'{p:.2e}' if p < 0.001 else f'{p:.3f}',
            'sig':        '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns',
        })

friedman_df = pd.DataFrame(friedman_results)
print(friedman_df.to_string(index=False))

In [ ]:
friedman_df['param_label'] = friedman_df['parameter'].map({
    'alpha_weight': 'Alpha Weight',
    'filter_size':  'Filter Size',
    'hist_percent': 'Hist Percent',
    'num_samples':  'Num Samples',
})

res_order   = ['100%', '87%', '71%', '50%']
param_order = ['Alpha Weight', 'Filter Size', 'Hist Percent', 'Num Samples']

friedman_df['neg_log10_p'] = friedman_df['p_value'].apply(
    lambda p: -np.log10(p) if p > 0 else 0
)
max_nlp = friedman_df['neg_log10_p'].max()

base = alt.Chart(friedman_df).encode(
    x=alt.X('param_label:N', sort=param_order, title=None,
             axis=alt.Axis(labelAngle=-30, labelFontSize=12)),
    y=alt.Y('resolution:N', sort=res_order, title=None,
             axis=alt.Axis(labelFontSize=12)),
)

rect = base.mark_rect().encode(
    color=alt.Color('neg_log10_p:Q',
                    scale=alt.Scale(domain=[0, max_nlp],
                                    range=['#f7f7f7', '#08519c']),
                    legend=None),
)

friedman_df['dark_bg'] = friedman_df['neg_log10_p'] > (max_nlp * 0.5)

text_sig = base.mark_text(fontSize=14, fontWeight='bold', dy=-7).encode(
    text='sig:N',
    color=alt.condition(alt.datum.dark_bg, alt.value('white'), alt.value('#222'))
)

text_p = base.mark_text(fontSize=10, dy=7).encode(
    text='p_display:N',
    color=alt.condition(alt.datum.dark_bg, alt.value('#ddd'), alt.value('#666'))
)

(rect + text_sig + text_p).properties(
    title=alt.TitleParams('Friedman Test — TAA Parameter Significance per Resolution',
                          fontSize=14, anchor='middle'),
    width=350, height=160
).configure_axis(
    labelFontSize=12
).configure_title(fontSize=13)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.stats import friedmanchisquare

# ── Compute Friedman + Kendall's W ───────────────────────────────────────────
friedman_results = []
for param in PARAMS:
    for res in [100,87,71,50]:
        sub = df[(df['parameter'] == param) & (df['resolution'] == res)]
        pivot = sub.pivot_table(index='scene', columns='value', values='score').dropna()
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            continue
        stat, p = friedmanchisquare(*[pivot[col].values for col in pivot.columns])
        n_scenes, n_levels = pivot.shape[0], pivot.shape[1]
        W = stat / (n_scenes * (n_levels - 1))
        friedman_results.append({
            'parameter':  param,
            'resolution': f'{res}%',
            'statistic':  round(stat, 3),
            'p_value':    p,
            'p_display':  f'{p:.2e}' if p < 0.001 else f'{p:.3f}',
            'sig':        '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns',
            'kendalls_W': round(W, 3),
            'W_display':  f'W={W:.2f}',
        })

fdf = pd.DataFrame(friedman_results)
fdf['param_label'] = fdf['parameter'].map(param_labels)
fdf['neg_log10_p'] = fdf['p_value'].apply(lambda p: -np.log10(p) if p > 0 else 0)
max_nlp = fdf['neg_log10_p'].max()

param_order = ['Alpha Weight', 'Filter Size', 'Hist Percent (100-200)', 'Num Samples']
res_order   = ['100%', '87%', '71%', '50%']

# ── Pivot into 2D grids ───────────────────────────────────────────────────────
def make_grid(fdf, value_col):
    return fdf.pivot_table(index='resolution', columns='param_label',
                           values=value_col, aggfunc='first').reindex(
                               index=res_order, columns=param_order)

grid_p   = make_grid(fdf, 'neg_log10_p')
grid_W   = make_grid(fdf, 'kendalls_W')
grid_sig = make_grid(fdf, 'sig')
grid_pd  = make_grid(fdf, 'p_display')
grid_Wd  = make_grid(fdf, 'W_display')

nrows, ncols = len(res_order), len(param_order)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.5))
fig.suptitle('Friedman Test: Significance and Effect Size per Parameter × Resolution',
             fontsize=13, y=1.02)

def draw_heatmap(ax, grid_vals, grid_text1, grid_text2,
                 cmap, vmin, vmax, title, cbar_label):
    data = grid_vals.values.astype(float)
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
    cbar.set_label(cbar_label, fontsize=10)

    ax.set_xticks(range(ncols))
    ax.set_xticklabels(param_order, fontsize=10, rotation=-30, ha='left')
    ax.set_yticks(range(nrows))
    ax.set_yticklabels(res_order, fontsize=10)
    ax.set_title(title, fontsize=11, pad=8)

    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    cmap_obj = plt.get_cmap(cmap)

    for i in range(nrows):
        for j in range(ncols):
            val = data[i, j]
            # luminance of the background cell
            rgba = cmap_obj(norm(val))
            lum  = 0.299*rgba[0] + 0.587*rgba[1] + 0.114*rgba[2]
            fg   = 'white' if lum < 0.5 else '#222222'
            fg2  = '#dddddd' if lum < 0.5 else '#666666'

            t1 = grid_text1.iloc[i, j]
            t2 = grid_text2.iloc[i, j]
            ax.text(j, i - 0.12, str(t1), ha='center', va='center',
                    fontsize=11, fontweight='bold', color=fg)
            ax.text(j, i + 0.22, str(t2), ha='center', va='center',
                    fontsize=8, color=fg2)

# p-value heatmap
draw_heatmap(ax1,
             grid_vals=grid_p,
             grid_text1=grid_sig,
             grid_text2=grid_pd,
             cmap='Blues',
             vmin=0, vmax=max_nlp,
             title='Significance  (−log10 p)',
             cbar_label='−log10 p')

# Kendall's W heatmap
draw_heatmap(ax2,
             grid_vals=grid_W,
             grid_text1=grid_Wd,
             grid_text2=grid_sig,
             cmap='Oranges',
             vmin=0, vmax=1,
             title="Effect Size  (Kendall's W)",
             cbar_label="Kendall's W")

plt.tight_layout()
plt.savefig('friedman_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# stats_tests.py  ·  Unified statistical testing under three centering schemes
# ══════════════════════════════════════════════════════════════════════════

import warnings
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM
import altair as alt

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────
# 1. CENTERING STRATEGIES
#    Each function takes the full df and returns a copy with a "score_c" column.
# ──────────────────────────────────────────────────────────────────────────

def center_shared_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Shared-value centering (your original 'score_centered').
    For each (scene, parameter, resolution), subtract the mean score computed
    only over values that are present for *all* scenes in that (parameter,
    resolution) slice.  This anchors comparisons to a common baseline and
    removes between-scene bias without assuming the full value grid is balanced.
    """
    def _center(group):
        param, res = group["parameter"].iloc[0], group["resolution"].iloc[0]
        slice_     = df[(df["parameter"] == param) & (df["resolution"] == res)]
        n_scenes   = slice_["scene"].nunique()
        shared     = slice_.groupby("value")["scene"].nunique()
        shared_vals = shared[shared == n_scenes].index
        mask = group["value"].isin(shared_vals)
        mean = group.loc[mask, "score"].mean()
        return group["score"] - mean

    out = df.copy()
    out["score_c"] = df.groupby(
        ["scene", "parameter", "resolution"], group_keys=False
    ).apply(_center)
    return out


def center_global(df: pd.DataFrame) -> pd.DataFrame:
    """
    Global (grand-mean) demeaning.
    Subtracts each scene's overall mean score and adds back the dataset-wide
    grand mean:  score_c = score - scene_mean + grand_mean.
    Removes between-scene level differences while preserving the original scale.
    """
    grand_mean  = df["score"].mean()
    scene_means = df.groupby("scene")["score"].mean()
    out = df.copy()
    out["score_c"] = df["score"] - df["scene"].map(scene_means) + grand_mean
    return out


def center_none(df: pd.DataFrame) -> pd.DataFrame:
    """
    No centering — raw scores passed through as-is.
    Between-scene variance is NOT removed; for rm-ANOVA / Spearman / Kruskal
    this inflates residuals and can distort p-values.  LME is unaffected
    because its random intercept absorbs scene-level bias regardless.
    """
    out = df.copy()
    out["score_c"] = df["score"]
    return out


# ──────────────────────────────────────────────────────────────────────────
# 2. TEST FUNCTIONS
#    Each receives a (parameter, resolution) slice of a *centered* DataFrame
#    (i.e. one that already has a "score_c" column) and returns a flat dict
#    of scalar results.  LME always uses the raw "score" column.
# ──────────────────────────────────────────────────────────────────────────

def _normalise(s: pd.Series) -> pd.Series:
    lo, hi = s.min(), s.max()
    return pd.Series(np.zeros(len(s)), index=s.index) if hi == lo else (s - lo) / (hi - lo)


def test_rm_anova(grp: pd.DataFrame) -> dict:
    """
    Type-II OLS F-test: score_c ~ C(value).
    Tests whether the mean centered score differs across discrete value levels.
    Centering removes between-scene variance so no scene term is needed.
    """
    try:
        model = ols("score_c ~ C(value)", data=grp).fit()
        tbl   = sm.stats.anova_lm(model, typ=2)
        row   = tbl.loc["C(value)"]
        return dict(F=float(row["F"]),
                    df_num=int(row["df"]),
                    df_den=int(tbl.loc["Residual", "df"]),
                    p=float(row["PR(>F)"]))
    except Exception as e:
        return dict(F=np.nan, df_num=np.nan, df_den=np.nan, p=np.nan, error=str(e))


def test_spearman(grp: pd.DataFrame) -> dict:
    """
    Spearman ρ between parameter value and score_c across all (scene × value) pairs.
    Tests for a monotonic relationship between value magnitude and centered score.
    """
    try:
        rho, p = stats.spearmanr(grp["value"], grp["score_c"])
        return dict(rho=float(rho), p=float(p), n=len(grp))
    except Exception as e:
        return dict(rho=np.nan, p=np.nan, n=len(grp), error=str(e))


def test_kruskal(grp: pd.DataFrame) -> dict:
    """
    Kruskal-Wallis H-test on score_c, grouped by value level.
    Nonparametric analog of rm-ANOVA; robust to non-normality and unbalanced
    value coverage across scenes.
    """
    try:
        groups = [g["score_c"].values for _, g in grp.groupby("value")]
        if len(groups) < 2:
            raise ValueError("Need ≥ 2 value levels.")
        H, p = stats.kruskal(*groups)
        return dict(H=float(H), df=len(groups) - 1, p=float(p))
    except Exception as e:
        return dict(H=np.nan, df=np.nan, p=np.nan, error=str(e))


def test_lme(grp: pd.DataFrame) -> dict:
    """
    Linear mixed-effects model on RAW scores: score ~ value_norm [+ value_norm²] + (1|scene).
    value is min-max normalised to [0,1] before fitting.
    Fits both linear and quadratic models (REML=False) then selects via LRT (α=0.05).
    The random scene intercept absorbs between-scene bias directly, so centering
    of the score column is irrelevant here.
    """
    try:
        g = grp.copy()
        g["value_norm"]  = _normalise(g["value"])
        g["value_norm2"] = g["value_norm"] ** 2

        fit_lin  = MixedLM.from_formula(
            "score ~ value_norm", data=g, groups=g["scene"]
        ).fit(reml=False, method="lbfgs")
        fit_quad = MixedLM.from_formula(
            "score ~ value_norm + value_norm2", data=g, groups=g["scene"]
        ).fit(reml=False, method="lbfgs")

        lr_stat = 2.0 * (fit_quad.llf - fit_lin.llf)
        lr_p    = float(stats.chi2.sf(lr_stat, df=1))
        best    = fit_quad if lr_p < 0.05 else fit_lin

        return dict(
            coef_linear    = float(best.params["value_norm"]),
            coef_quadratic = float(best.params.get("value_norm2", np.nan)),
            p              = float(best.pvalues["value_norm"]),
            lr_p           = lr_p,
            model          = "quadratic" if lr_p < 0.05 else "linear",
        )
    except Exception as e:
        return dict(coef_linear=np.nan, coef_quadratic=np.nan,
                    p=np.nan, lr_p=np.nan, model="error", error=str(e))


TEST_FNS = dict(rm_anova=test_rm_anova, spearman=test_spearman,
                kruskal=test_kruskal,   lme=test_lme)


# ──────────────────────────────────────────────────────────────────────────
# 3. RUN PIPELINE
#    Accepts a centered DataFrame; returns a tidy results DataFrame.
# ──────────────────────────────────────────────────────────────────────────

def run_tests(df_centered: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (param, res), sdf in df_centered.groupby(["parameter", "resolution"]):
        row = dict(parameter=param, resolution=res)
        for name, fn in TEST_FNS.items():
            for k, v in fn(sdf).items():
                row[f"{name}_{k}"] = v
        rows.append(row)
    return (pd.DataFrame(rows)
            .sort_values(["parameter", "resolution"])
            .reset_index(drop=True))


# ──────────────────────────────────────────────────────────────────────────
# 4. HEATMAP
# ──────────────────────────────────────────────────────────────────────────

RESOLUTION_ORDER = ["100%", "87%", "71%", "50%"]
PARAM_LABELS     = {"alpha_weight": "α weight",
                    "filter_size":  "filter size",
                    "hist_percent": "hist %",
                    "num_samples":  "num samples"}
PLOT_SPECS = [
    ("rm_anova_p", "ANOVA",       "rm_anova_F"),
    ("spearman_p", "Spearman",        "spearman_rho"),
    ("kruskal_p",  "Kruskal-Wallis",  "kruskal_H"),
    ("lme_p",      "LME",             "lme_coef_linear"),
]

def _sig_stars(p: float) -> str:
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def build_heatmap(results: pd.DataFrame, subtitle: str) -> alt.Chart:
    # Build long-form plot table
    plot_rows = []
    for _, r in results.iterrows():
        for p_col, label, stat_col in PLOT_SPECS:
            p_val  = r.get(p_col,    np.nan)
            stat_v = r.get(stat_col, np.nan)
            neg_log = (-np.log10(p_val)
                       if (p_val is not None and p_val > 0 and not np.isnan(p_val))
                       else 0.0)
            plot_rows.append(dict(
                test       = label,
                parameter  = PARAM_LABELS.get(r["parameter"], r["parameter"]),
                resolution = f"{r['resolution']}%",
                neg_log_p  = neg_log,
                p_val      = p_val,
                stat_val   = stat_v,
                stars      = _sig_stars(p_val) if not np.isnan(p_val) else "",
            ))
    plot_df = pd.DataFrame(plot_rows)

    thresh      = -np.log10(0.05)
    max_nlp     = plot_df["neg_log_p"].replace([np.inf], np.nan).max()
    clamp       = min(max_nlp * 1.05, 10)
    WHITE_THRESH = (thresh + clamp) / 2

    color_scale = alt.Scale(
        domain=[0, thresh, clamp],
        range=["#f0f0f0", "#fdcc8a", "#b30000"],
    )

    def make_panel(test_label: str) -> alt.Chart:
        base = (alt.Chart(plot_df)
            .transform_filter(alt.datum.test == test_label)
            .encode(
                x=alt.X("parameter:O",
                         sort=list(PARAM_LABELS.values()),
                         axis=alt.Axis(title=None, labelAngle=-25)),
                y=alt.Y("resolution:O",
                         sort=RESOLUTION_ORDER,
                         axis=alt.Axis(title="Resolution")),
            ))

        text_color = alt.condition(
            alt.datum.neg_log_p > WHITE_THRESH,
            alt.value("white"), alt.value("#222222"),
        )
        return (
            base.mark_rect().encode(
                color=alt.Color("neg_log_p:Q", scale=color_scale,
                                legend=alt.Legend(title="-log₁₀(p)",
                                                  orient="right",
                                                  gradientLength=120)),
                tooltip=[
                    alt.Tooltip("parameter:O",  title="Parameter"),
                    alt.Tooltip("resolution:O", title="Resolution"),
                    alt.Tooltip("p_val:Q",      title="p-value",    format=".4f"),
                    alt.Tooltip("neg_log_p:Q",  title="-log₁₀(p)",  format=".2f"),
                    alt.Tooltip("stat_val:Q",   title="Statistic",  format=".3f"),
                ],
            )
            + base.mark_text(fontSize=13, fontWeight="bold", dy=6).encode(
                text="stars:N", color=text_color)
            + base.mark_text(fontSize=9, dy=-5).encode(
                text=alt.condition(alt.datum.stat_val != None,
                                   alt.Text("stat_val:Q", format=".2f"),
                                   alt.value("")),
                color=text_color,
            )
        ).properties(
            title=alt.TitleParams(test_label, fontSize=13,
                                  fontWeight="bold", anchor="middle"),
            width=200, height=130,
        )

    return (
        alt.hconcat(*[make_panel(label) for _, label, _ in PLOT_SPECS], spacing=18)
        .properties(title=alt.TitleParams(
            ["CGVQM Statistical Tests — −log₁₀(p) per (parameter × resolution)",
             subtitle],
            fontSize=14, anchor="middle", dy=-6,
        ))
        .configure_view(strokeWidth=0)
        .configure_axis(labelFontSize=11, titleFontSize=11)
        .configure_title(fontSize=14)
    )


# ──────────────────────────────────────────────────────────────────────────
# 5. EXECUTE ALL THREE CONDITIONS
# ──────────────────────────────────────────────────────────────────────────

CENTERING_CONDITIONS = [
    (center_shared_values, "Centering: q̃ᵥ = qᵥ − q̄ₛ꜀  (shared-value baseline)"),
    (center_global,        "Centering: q̃ = q − scene mean + grand mean"),
    (center_none,          "Centering: none (raw scores)"),
]

all_results = {}
for center_fn, subtitle in CENTERING_CONDITIONS:
    name       = center_fn.__name__          # e.g. "center_shared_values"
    df_c       = center_fn(df)
    results    = run_tests(df_c)
    all_results[name] = results

    _cols = ["parameter", "resolution",
             "rm_anova_F", "rm_anova_p",
             "spearman_rho", "spearman_p",
             "kruskal_H", "kruskal_p",
             "lme_coef_linear", "lme_p", "lme_model"]
    print(f"\n{'═'*70}")
    print(f"  {subtitle}")
    print('═'*70)
    print(results[_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

    build_heatmap(results, subtitle).display()

In [ ]:
# Quick effect size check for hist_percent
for res in [50, 71, 87, 100]:
    sdf = df[(df.parameter=="hist_percent") & (df.resolution==res)]
    group_means = sdf.groupby("value")["score_centered"].mean()
    spread = group_means.max() - group_means.min()
    overall_std = sdf["score_centered"].std()
    print(f"{res}%  mean spread across values: {spread:.3f}  std: {overall_std:.3f}  ratio: {spread/overall_std:.2f}")

In [ ]:
# ── Corrected Sanity Check ─────────────────────────────────────────────────

# 1. For each (scene, parameter, resolution), compute mean of score_centered
#    over ONLY the shared values (present in ALL scenes for that param×res).
errors = []
for (param, res), sdf in df.groupby(["parameter", "resolution"]):
    n_scenes = sdf["scene"].nunique()
    shared_vals = (sdf.groupby("value")["scene"].nunique()
                     .pipe(lambda s: s[s == n_scenes].index))
    for scene, scenedf in sdf.groupby("scene"):
        mask = scenedf["value"].isin(shared_vals)
        mean_centered = scenedf.loc[mask, "score_centered"].mean()
        errors.append(dict(scene=scene, parameter=param,
                           resolution=res, mean=mean_centered))

err_df = pd.DataFrame(errors)
print("Max absolute within-group mean over shared values (should be ~0):",
      err_df["mean"].abs().max().round(8))
print("Any group with |mean| > 1e-6:",
      (err_df["mean"].abs() > 1e-6).sum(), "of", len(err_df))

# 2. Variance reduction check (unchanged — this one was fine)
print("\nRaw score std:      ", df["score"].std().round(4))
print("Centered score std: ", df["score_centered"].std().round(4))

# 3. Spot-check (unchanged — this one was fine)
scene, param, res = df["scene"].iloc[0], "alpha_weight", 100
sub = df[(df.scene==scene)&(df.parameter==param)&(df.resolution==res)]
print(f"\nSpot check — {scene}, {param}, {res}%:")
print(sub[["value","score","score_centered"]].to_string(index=False))
print("Mean of score_centered:", sub["score_centered"].mean().round(8))

### Alpha weight: peak value distribution per resolution

In [ ]:
peak_hist = alt.Chart(peaks).mark_bar(opacity=0.7).encode(
    x=alt.X('peak_value:Q', bin=alt.Bin(step=0.1), title='Peak alpha_weight value'),
    y=alt.Y('count():Q', title='Number of scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=150, height=150, title='Distribution of Optimal alpha_weight Per Scene')

peak_hist

In [ ]:
from numpy.polynomial import polynomial as P

quad_records = []

for (res), group in alpha_df.groupby('resolution'):
    # Aggregate to mean per value across scenes
    curve = group.groupby('value')['score_centered'].mean()
    x = curve.index.values
    y = curve.values

    # Fit quadratic
    coeffs = np.polyfit(x, y, 2)  # [a, b, c] for ax² + bx + c
    a, b, c = coeffs
    peak_x = -b / (2 * a)  # vertex of parabola

    # R² 
    y_pred = np.polyval(coeffs, x)
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot

    quad_records.append({
        'resolution': res,
        'a (quadratic)': round(a, 4),
        'b (linear)': round(b, 4),
        'c (intercept)': round(c, 4),
        'peak_at': round(peak_x, 3),
        'R²': round(r2, 4),
        'concave_down': a < 0
    })

    # Plot fitted curve
    x_smooth = np.linspace(x.min(), x.max(), 100)
    y_smooth = np.polyval(coeffs, x_smooth)
    fit_df = pd.DataFrame({'value': x_smooth, 'fitted': y_smooth, 'resolution': res})
    quad_records_plot = fit_df if res == res_order[0] else pd.concat([quad_records_plot, fit_df])
    if res == res_order[0]:
        quad_records_plot = fit_df
    else:
        quad_records_plot = pd.concat([quad_records_plot, fit_df])

quad_summary = pd.DataFrame(quad_records)
print(quad_summary.to_string(index=False))

###  hist_percent: per-scene Spearman r + range

In [ ]:
hist_df = df[df['parameter'] == 'hist_percent'].copy()

spearman_records = []
range_records = []

for (scene, res), group in hist_df.groupby(['scene', 'resolution']):
    val_means = group.groupby('value')['score_centered'].mean()
    if len(val_means) < 3:
        continue
    r, p = stats.spearmanr(val_means.index, val_means.values)
    score_range = group.groupby('value')['score'].mean().max() - group.groupby('value')['score'].mean().min()
    spearman_records.append({'scene': scene, 'resolution': res, 'spearman_r': r, 'p_value': p})
    range_records.append({'scene': scene, 'resolution': res, 'score_range': score_range})

spearman_hist = pd.DataFrame(spearman_records)
range_hist = pd.DataFrame(range_records)
spearman_hist['resolution'] = spearman_hist['resolution'].astype(str) + '%'
range_hist['resolution'] = range_hist['resolution'].astype(str) + '%'

# Spearman r distribution
r_plot = alt.Chart(spearman_hist).mark_bar(opacity=0.7).encode(
    x=alt.X('spearman_r:Q', bin=alt.Bin(step=0.2), title='Spearman r'),
    y=alt.Y('count():Q', title='Scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=130, height=150, title='hist_percent: Per-Scene Spearman r (monotonic consistency)')

# Score range distribution
range_plot = alt.Chart(range_hist).mark_bar(opacity=0.7).encode(
    x=alt.X('score_range:Q', bin=alt.Bin(step=0.5), title='Score range (pts)'),
    y=alt.Y('count():Q', title='Scenes'),
    color=alt.Color('resolution:N', scale=color_scale, sort=res_order),
    column=alt.Column('resolution:N', sort=res_order)
).properties(width=130, height=150, title='hist_percent: Per-Scene Score Range vs 5pt threshold')

# Reference line at 5 pts
threshold_line = alt.Chart(pd.DataFrame({'x': [5]})).mark_rule(
    color='red', strokeDash=[4,4]
).encode(x='x:Q')

r_plot & (range_plot + threshold_line)

In [ ]:
from scipy.stats import ttest_1samp

EQUIVALENCE_BOUND = 2.0  # points — half the 5pt perceptual threshold, conservative

tost_records = []

for param in ['filter_size', 'num_samples']:
    for (res), group in df[df['parameter'] == param].groupby('resolution'):
        # Per scene: compute range across parameter values
        scene_ranges = group.groupby('scene').apply(
            lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
        )
        # TOST: test that mean range is less than EQUIVALENCE_BOUND
        # H0: effect >= bound  →  reject = effect is negligible
        t_upper, p_upper = ttest_1samp(scene_ranges, EQUIVALENCE_BOUND)
        t_lower, p_lower = ttest_1samp(scene_ranges, -EQUIVALENCE_BOUND)
        tost_p = max(p_upper / 2, p_lower / 2)  # one-tailed
        tost_records.append({
            'parameter': param,
            'resolution': res,
            'mean_range': round(scene_ranges.mean(), 4),
            'std_range': round(scene_ranges.std(), 4),
            'TOST_p': round(tost_p, 6),
            'equivalent': tost_p < 0.05,
            'bound': EQUIVALENCE_BOUND
        })

tost_df = pd.DataFrame(tost_records)
print(tost_df.to_string(index=False))

In [ ]:
flat_params = ['filter_size', 'num_samples']
agg_flat = df[df['parameter'].isin(flat_params)].groupby(
    ['parameter', 'resolution', 'value'])['score_centered'].agg(['mean','std','count']).reset_index()
agg_flat['ci95'] = 1.96 * agg_flat['std'] / np.sqrt(agg_flat['count'])
agg_flat['resolution'] = agg_flat['resolution'].astype(str) + '%'

# Equivalence band (±2 pts)
band_df = pd.DataFrame({'y': [EQUIVALENCE_BOUND], 'y2': [-EQUIVALENCE_BOUND]})
eq_band = alt.Chart(band_df).mark_rect(opacity=0.08, color='gray').encode(
    y='y2:Q', y2='y:Q'
)
eq_upper = alt.Chart(band_df).mark_rule(strokeDash=[4,4], color='gray', opacity=0.5).encode(y='y:Q')
eq_lower = alt.Chart(band_df).mark_rule(strokeDash=[4,4], color='gray', opacity=0.5).encode(y='y2:Q')

charts = []
for param in flat_params:
    sub = agg_flat[agg_flat['parameter'] == param]
    line = alt.Chart(sub).mark_line(point=True).encode(
        x=alt.X('value:Q', title=param),
        y=alt.Y('mean:Q', title='Centered CGVQM', scale=alt.Scale(domain=[-3, 3])),
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    band = alt.Chart(sub).mark_errorband(opacity=0.15).encode(
        x='value:Q',
        y=alt.Y('mean:Q', scale=alt.Scale(domain=[-3, 3])),
        yError='ci95:Q',
        color=alt.Color('resolution:N', scale=color_scale, sort=res_order)
    )
    charts.append((eq_band + eq_upper + eq_lower + band + line).properties(
        title=param, width=300, height=220
    ))

alt.hconcat(*charts).resolve_scale(color='shared').properties(
    title=f'filter_size & num_samples: Effects Within ±{EQUIVALENCE_BOUND}pt Equivalence Band'
)

### Statistical Tests 
The issue with using some of these tests is that they want some sort of monotonic increase--this is not what happens with alpha weight. So, they often show significance weirdly...

#### Anova Test (said everything was insignificant)

In [ ]:
# Anova shows everything as insignificant...
anova_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    groups_by_value = [g['score'].values for _, g in group.groupby('value')]
    if len(groups_by_value) < 2:
        continue
    f_stat, p_val = stats.f_oneway(*groups_by_value)
    anova_records.append({
        'parameter': param,
        'resolution': res,
        'F': round(f_stat, 4),
        'p_value': round(p_val, 6),
        'significant': p_val < 0.05,
    })

anova_df = pd.DataFrame(anova_records)
anova_df

In [ ]:
effect_records = []

for (param, res), group in df.groupby(['parameter', 'resolution']):
    scene_ranges = group.groupby('scene').apply(
        lambda s: s.groupby('value')['score'].mean().max() - s.groupby('value')['score'].mean().min()
    )
    effect_records.append({
        'parameter': param,
        'resolution': res,
        'mean_range': scene_ranges.mean(),
        'se_range': scene_ranges.sem()
    })

effect_df = pd.DataFrame(effect_records)
effect_df['range_label'] = effect_df['mean_range'].apply(lambda x: f'{x:.2f}')

In [ ]:
from scipy.stats import friedmanchisquare

# ── Friedman ──────────────────────────────────────────────────────────────────
def compute_friedman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        values = sorted(group['value'].unique())
        groups_by_value = [group[group['value'] == v]['score_centered'].values for v in values]
        if any(len(g) < 3 for g in groups_by_value):
            continue
        _, p = friedmanchisquare(*groups_by_value)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'test': 'Friedman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Kendall's W ───────────────────────────────────────────────────────────────
def kendalls_w(data):
    n, k = data.shape
    ranks = data.rank(axis=1)
    rank_sums = ranks.sum(axis=0)
    S = ((rank_sums - rank_sums.mean()) ** 2).sum()
    W = 12 * S / (n ** 2 * (k ** 3 - k))
    chi2 = n * (k - 1) * W
    p = 1 - stats.chi2.cdf(chi2, df=k - 1)
    return W, p

def compute_kendall(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        value_counts = group.groupby('scene')['value'].nunique()
        n_values = group['value'].nunique()
        complete_scenes = value_counts[value_counts == n_values].index
        group = group[group['scene'].isin(complete_scenes)]
        if group['scene'].nunique() < 3:
            continue
        pivot = group.pivot_table(index='scene', columns='value', values='score_centered').dropna()
        if pivot.shape[0] < 3 or pivot.shape[1] < 2:
            continue
        W, p = kendalls_w(pivot)
        records.append({'parameter': param, 'resolution': res, 'p_value': p, 'W': round(W, 4), 'test': 'Kendall W'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

# ── Spearman ──────────────────────────────────────────────────────────────────
def compute_spearman(df):
    records = []
    for (param, res), group in df.groupby(['parameter', 'resolution']):
        scene_corrs = []
        for scene, sgroup in group.groupby('scene'):
            val_means = sgroup.groupby('value')['score_centered'].mean()
            if len(val_means) < 3:
                continue
            r, _ = stats.spearmanr(val_means.index, val_means.values)
            scene_corrs.append(r)
        if len(scene_corrs) < 3:
            continue
        t, p = stats.ttest_1samp(scene_corrs, 0)
        records.append({'parameter': param, 'resolution': res, 'mean_r': round(np.mean(scene_corrs), 4), 'p_value': round(p, 6), 'test': 'Spearman'})
    out = pd.DataFrame(records)
    out['significant'] = out['p_value'] < 0.05
    return out

In [ ]:
# ── Compute all ───────────────────────────────────────────────────────────────
sig_friedman = compute_friedman(df)
sig_kendall  = compute_kendall(df)
sig_spearman = compute_spearman(df)

In [ ]:
# ── Toggle this to switch tests in all downstream plots ───────────────────────
sig_df = sig_spearman   # ← change to sig_friedman, sig_kendall, or sig_spearman
print(sig_df)

In [ ]:
summary_df = effect_df.merge(sig_df[['parameter', 'resolution', 'p_value', 'significant', 'test']], on=['parameter', 'resolution'])

base = alt.Chart(summary_df)

heatmap = base.mark_rect().encode(
    x=alt.X('resolution:O', title='Base Resolution', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N', title='TAA Parameter'),
    color=alt.Color('mean_range:Q',
                    scale=alt.Scale(scheme='blues', domain=[0, 5]),
                    title='Mean Score Range (pts)'),
    tooltip=['parameter', 'resolution',
             alt.Tooltip('mean_range:Q', format='.3f'),
             alt.Tooltip('p_value:Q', format='.4f'),
             'significant', 'test']
)

range_text = base.mark_text(fontSize=11, dy=-6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text='range_label:N',
    color=alt.condition(alt.datum.mean_range > 2.5, alt.value('white'), alt.value('black'))
)

sig_text = base.mark_text(fontSize=14, dy=6).encode(
    x=alt.X('resolution:O', sort=[50, 71, 87, 100]),
    y=alt.Y('parameter:N'),
    text=alt.condition(alt.datum.significant, alt.value('✱'), alt.value('')),
    color=alt.condition(alt.datum.mean_range > 2.5, alt.value('white'), alt.value('black'))
)

(heatmap + range_text + sig_text).properties(
    width=300, height=200,
    title=f'TAA Parameter Effect × Resolution ({summary_df["test"].iloc[0]})'
)

In [ ]:
# Do scenes agree on the direction of the effect for each parameter?
for param in PARAMS:
    sub = df[df['parameter'] == param]
    scene_corrs = []
    for scene, sgroup in sub.groupby('scene'):
        val_means = sgroup.groupby('value')['score_centered'].mean()
        if len(val_means) < 3:
            continue
        r, _ = stats.spearmanr(val_means.index, val_means.values)
        scene_corrs.append(r)
    print(f"{param}: mean_r={np.mean(scene_corrs):.3f}, std_r={np.std(scene_corrs):.3f}, n={len(scene_corrs)}")